## Spatial Feature Engineering

### Purpose

Transform raw crime incident records into community-area level spatial features that summarize historical crime patterns.

### Feature Meaning

| Feature | Description |
|--------|-------------|
| crime_count_last3y | Total number of crime incidents in the community area during the previous three years. |
| lat_mean | Average latitude of crime incidents in the community area, representing the geographic center of crime activity. |
| lon_mean | Average longitude of crime incidents in the community area, representing the geographic center of crime activity. |
| district_mode | The most frequently occurring police district within the community area. |
| location_type_ratios | Proportions of different location categories (e.g., commercial, residential, public outdoor, institution) where crimes occurred. |
| crime_type_ratios | Proportions of major crime categories (e.g., theft, battery, assault, etc.) within the community area. |

### Output

| Dataset Type       | Name           | Description |
|--------------------|---------------|-------------|
| Training Dataset   | training_dataset | Community-area level dataset generated from the 2015–2024 crime data using rolling windows. Features represent crime patterns in the previous three years and the label indicates whether the area is a hotspot in the following year. |
| Training Features  | X_train             | Final feature matrix after feature engineering, one-hot encoding (district), and normalization, used to train the models. |
| Training Labels    | y_train             | Binary hotspot labels corresponding to the training samples. |
| Test Features      | X_test  | Feature dataset constructed using crime data from 2022–2024 to predict hotspot areas for 2025. |
| Test Labels        | y_test        | Ground truth hotspot labels for 2025, generated using the top 25% crime count rule across community areas. |

In [1]:
import pandas as pd

train = pd.read_csv("../data/furtherprocessed/chicago_crimes_2015_2024_processed.csv")
test = pd.read_csv("../data/furtherprocessed/chicago_crimes_2025_processed.csv")

In [2]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

In [3]:
train["primary_type"] = train["primary_type"].astype("category")
train["location_description"] = train["location_description"].astype("category")

test["primary_type"] = test["primary_type"].astype("category")
test["location_description"] = test["location_description"].astype("category")

# Convert community_area to integer
train["community_area"] = train["community_area"].astype(int)
test["community_area"] = test["community_area"].astype(int)

#Convert district to  integer
train["district"] = train["district"].astype(int)
test["district"] = test["district"].astype(int)

In [4]:
print(train.shape)
print(train.columns)

train.info()

train.head()

(2424661, 15)
Index(['date', 'primary_type', 'location_description', 'arrest', 'beat',
       'district', 'ward', 'community_area', 'latitude', 'longitude',
       'loc_COMMERCIAL', 'loc_INSTITUTION', 'loc_OTHER', 'loc_PUBLIC_OUTDOOR',
       'loc_RESIDENTIAL'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2424661 entries, 0 to 2424660
Data columns (total 15 columns):
 #   Column                Dtype         
---  ------                -----         
 0   date                  datetime64[ns]
 1   primary_type          category      
 2   location_description  category      
 3   arrest                int64         
 4   beat                  int64         
 5   district              int32         
 6   ward                  float64       
 7   community_area        int32         
 8   latitude              float64       
 9   longitude             float64       
 10  loc_COMMERCIAL        int64         
 11  loc_INSTITUTION       int64         
 12  loc_OTHER

,date,primary_type,location_description,arrest,beat,district,ward,community_area,latitude,longitude,loc_COMMERCIAL,loc_INSTITUTION,loc_OTHER,loc_PUBLIC_OUTDOOR,loc_RESIDENTIAL
0,2015-03-19 16:47:00,BATTERY,STREET,1,2535,25,26.0,23,41.906354,-87.718554,0,0,0,1,0
1,2015-03-22 02:34:00,BATTERY,HOTEL/MOTEL,1,833,8,13.0,65,41.759284,-87.741709,0,0,1,0,0
2,2015-03-26 15:45:00,THEFT,SMALL RETAIL STORE,1,2422,24,49.0,1,42.019399,-87.675049,0,0,1,0,0
3,2015-03-31 12:00:00,BURGLARY,RESIDENCE,0,2211,22,19.0,74,41.683936,-87.721847,0,0,0,0,1
4,2015-03-31 20:41:00,BATTERY,APARTMENT,1,935,9,3.0,61,41.795248,-87.643360,0,0,0,0,1


Define major crime types

In [5]:
train["primary_type"].unique()

['BATTERY', 'THEFT', 'BURGLARY', 'ROBBERY', 'SEX OFFENSE', ..., 'INTIMIDATION', 'NON-CRIMINAL', 'OTHER NARCOTIC VIOLATION', 'HUMAN TRAFFICKING', 'RITUALISM']
Length: 33
Categories (33, object): ['ARSON', 'ASSAULT', 'BATTERY', 'BURGLARY', ..., 'SEX OFFENSE', 'STALKING', 'THEFT', 'WEAPONS VIOLATION']

In [6]:
top_crimes = train["primary_type"].value_counts().head(5).index.tolist()

print(top_crimes)

['THEFT', 'BATTERY', 'CRIMINAL DAMAGE', 'ASSAULT', 'DECEPTIVE PRACTICE']


### Feautre engineering function

In [7]:
def build_area_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate community_area-level features from crime event records.
    Each row in the output represents one community area.
    """

    # 1. Basic aggregated features

    features = df.groupby("community_area").agg(
        crime_count_last3y=("community_area", "size"),
        lat_mean=("latitude", "mean"),
        lon_mean=("longitude", "mean"),
        district_mode=("district", lambda x: x.mode().iloc[0]),
        #beat_mode=("beat", lambda x: x.mode().iloc[0]),
        commercial_ratio=("loc_COMMERCIAL", "mean"),
        institution_ratio=("loc_INSTITUTION", "mean"),
        other_ratio=("loc_OTHER", "mean"),
        public_ratio=("loc_PUBLIC_OUTDOOR", "mean"),
        residential_ratio=("loc_RESIDENTIAL", "mean"),
    ).reset_index()


    # 2. Crime type ratios

    crime_type_counts = pd.pivot_table(
        df,
        index="community_area",
        columns="primary_type",
        aggfunc="size",
        fill_value=0
    )    
    
    crime_top = crime_type_counts[top_crimes]

    crime_total_counts = crime_type_counts.sum(axis=1)

    crime_top_counts = crime_top.sum(axis=1)

    # ratio for top crimes
    crime_top_ratio = crime_top.div(crime_total_counts, axis=0)

    # ratio for remaining crimes
    other_crime_ratio = (crime_total_counts - crime_top_counts) / crime_total_counts

    # add other crime ratio
    crime_top_ratio["OTHER_CRIME"] = other_crime_ratio

    # Convert community_area from index back to a column so it can be used for merging later
    crime_type_ratio = crime_top_ratio.reset_index()

    # rename columns
    crime_type_ratio = crime_type_ratio.rename(columns={
        "THEFT": "theft_ratio",
        "BATTERY": "battery_ratio",
        "CRIMINAL DAMAGE": "criminal_damage_ratio",
        "ASSAULT": "assault_ratio",
        "DECEPTIVE PRACTICE": "deceptive_practice_ratio",
        "OTHER_CRIME": "other_crime_ratio"
    })

    # 3. Merge all features
    
    features = features.merge(crime_type_ratio, on="community_area", how="left")

    return features

### 3 year rolling window

In [8]:
training_rows = []

# Rolling window: use previous 3 years as features and next year as label
for target_year in range(2018, 2025):

    # Define feature window (previous 3 years)
    start_year = target_year - 3
    end_year = target_year - 1

    df_window = train[
        (train["date"].dt.year >= start_year) &
        (train["date"].dt.year <= end_year)
    ]

    # Build community_area level features
    X_part = build_area_features(df_window)

    # Create hotspot label
    df_target = train[train["date"].dt.year == target_year]

    crime_counts = (
        df_target.groupby("community_area")
        .size()
        .reset_index(name="crime_count")
    )

    # Define hotspot threshold (top 25%)
    threshold = crime_counts["crime_count"].quantile(0.75)

    crime_counts["hotspot"] = (crime_counts["crime_count"] >= threshold).astype(int)

    # Keep only label column
    y_part = crime_counts[["community_area", "hotspot"]]

    # Merge features and labels
    dataset_part = X_part.merge(y_part, on="community_area", how="left")

    # Store dataset for this window
    training_rows.append(dataset_part)

# Combine all windows
training_dataset = pd.concat(training_rows, ignore_index=True)


C:\Users\Dolores\AppData\Local\Temp\ipykernel_18248\2730137192.py:25: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  crime_type_counts = pd.pivot_table(
C:\Users\Dolores\AppData\Local\Temp\ipykernel_18248\2730137192.py:25: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  crime_type_counts = pd.pivot_table(
C:\Users\Dolores\AppData\Local\Temp\ipykernel_18248\2730137192.py:25: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  crime_type_counts = pd.pivot_table(
C:\Users\Dolores\AppData\Local\Temp\ipykernel_18248\

In [9]:
print(training_dataset.shape)
training_dataset.head(77)

(539, 17)


,community_area,crime_count_last3y,lat_mean,lon_mean,district_mode,commercial_ratio,institution_ratio,other_ratio,public_ratio,residential_ratio,theft_ratio,battery_ratio,criminal_damage_ratio,assault_ratio,deceptive_practice_ratio,other_crime_ratio,hotspot
0,1,9817,42.010972,-87.670587,24,0.024346,0.0,0.353061,0.305898,0.316696,0.275542,0.188449,0.116023,0.065193,0.063971,0.290822,0
1,2,9649,42.000055,-87.692958,24,0.020002,0.0,0.322624,0.327702,0.329671,0.232874,0.173386,0.138460,0.062804,0.073168,0.319308,0
2,3,10484,41.966214,-87.656776,19,0.028901,0.0,0.396414,0.305990,0.268695,0.264975,0.190195,0.088611,0.071919,0.095670,0.288630,0
3,4,5565,41.972195,-87.688435,20,0.026954,0.0,0.424438,0.288949,0.259659,0.286433,0.154897,0.119497,0.056783,0.103145,0.279245,0
4,5,4077,41.947263,-87.683124,19,0.024283,0.0,0.447633,0.302674,0.225411,0.361050,0.090753,0.111602,0.041697,0.118960,0.275938,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,73,9190,41.719136,-87.649859,22,0.039064,0.0,0.265506,0.326224,0.369206,0.175843,0.192492,0.132644,0.074646,0.058977,0.365397,0
73,74,1746,41.694971,-87.708258,22,0.017182,0.0,0.387743,0.215922,0.379152,0.263459,0.160367,0.117411,0.074456,0.122566,0.261741,0
74,75,5955,41.689276,-87.663029,22,0.035432,0.0,0.359866,0.267674,0.337028,0.224685,0.187070,0.125105,0.079093,0.079597,0.304450,0
75,76,4773,41.976388,-87.885561,16,0.003143,0.0,0.848523,0.052378,0.095956,0.304211,0.107061,0.064320,0.048607,0.128431,0.347371,0


In [10]:
training_dataset["district_mode"].unique()

array([24, 19, 20, 18, 16, 17, 25, 14, 11, 12, 15, 10,  1,  9,  2,  3,  6,
        4,  5,  8,  7, 22])

In [11]:
# One-hot encode district_mode
training_dataset = pd.get_dummies(
    training_dataset,
    columns=["district_mode"],
    prefix="district",
    dtype=int
)

In [12]:
X_train = training_dataset.drop(columns=["community_area", "hotspot"])

# X_raw: after one-hot encoding, before normalization
X_raw = training_dataset.drop(columns=["community_area", "hotspot"])

y_train = training_dataset["hotspot"]

### Normalization

In [13]:
from sklearn.preprocessing import MinMaxScaler

num_features = [
    "crime_count_last3y",
    "lat_mean",
    "lon_mean"
]

scaler = MinMaxScaler()

# training data
X_train[num_features] = scaler.fit_transform(X_train[num_features])

In [14]:
X_train.head()

,crime_count_last3y,lat_mean,lon_mean,commercial_ratio,institution_ratio,other_ratio,public_ratio,residential_ratio,theft_ratio,battery_ratio,...,district_14,district_15,district_16,district_17,district_18,district_19,district_20,district_22,district_24,district_25
0,0.191194,0.999955,0.613458,0.024346,0.0,0.353061,0.305898,0.316696,0.275542,0.188449,...,0,0,0,0,0,0,0,0,1,0
1,0.187655,0.969214,0.549667,0.020002,0.0,0.322624,0.327702,0.329671,0.232874,0.173386,...,0,0,0,0,0,0,0,0,1,0
2,0.205245,0.873924,0.652839,0.028901,0.0,0.396414,0.305990,0.268695,0.264975,0.190195,...,0,0,0,0,0,1,0,0,0,0
3,0.101622,0.890765,0.562563,0.026954,0.0,0.424438,0.288949,0.259659,0.286433,0.154897,...,0,0,0,0,0,0,1,0,0,0
4,0.070276,0.820561,0.577708,0.024283,0.0,0.447633,0.302674,0.225411,0.361050,0.090753,...,0,0,0,0,0,1,0,0,0,0


In [22]:
y_train.head(77)

0     0
1     0
2     0
3     0
4     0
     ..
72    0
73    0
74    0
75    0
76    0
Name: hotspot, Length: 77, dtype: int32

In [17]:
# Build test features
df_window_test = train[
    (train["date"].dt.year >= 2022) &
    (train["date"].dt.year <= 2024)
].copy()

X_test_raw = build_area_features(df_window_test)

# One-hot encode district
X_test_raw = pd.get_dummies(
    X_test_raw,
    columns=["district_mode"],
    prefix="district",
    dtype=int
)

# Align with training columns BEFORE scaling
X_raw_with_id = training_dataset.drop(columns=["hotspot"])
X_test_raw = X_raw_with_id.align(X_test_raw, join="left", axis=1, fill_value=0)[1]

# Scale numerical features
X_test_raw[num_features] = scaler.transform(X_test_raw[num_features])

C:\Users\Dolores\AppData\Local\Temp\ipykernel_18248\2730137192.py:25: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  crime_type_counts = pd.pivot_table(


In [18]:
# Build y_test
crime_counts_2025 = (
    test.groupby("community_area")
    .size()
    .reset_index(name="crime_count")
)

threshold_2025 = crime_counts_2025["crime_count"].quantile(0.75)
crime_counts_2025["hotspot"] = (
    crime_counts_2025["crime_count"] >= threshold_2025
).astype(int)

y_test_df = crime_counts_2025[["community_area", "hotspot"]]

In [19]:
# Merge feature and label
test_final = X_test_raw.merge(y_test_df, on="community_area", how="inner")

X_test = test_final.drop(columns=["community_area", "hotspot"])
y_test = test_final["hotspot"]

In [20]:
X_test.head()

,crime_count_last3y,lat_mean,lon_mean,commercial_ratio,institution_ratio,other_ratio,public_ratio,residential_ratio,theft_ratio,battery_ratio,...,district_14,district_15,district_16,district_17,district_18,district_19,district_20,district_22,district_24,district_25
0,0.228671,0.997561,0.615968,0.023974,0.0,0.350897,0.308123,0.317006,0.303036,0.177303,...,0,0,0,0,0,0,0,0,1,0
1,0.224057,0.968518,0.549710,0.022414,0.0,0.274237,0.356685,0.346664,0.231608,0.162785,...,0,0,0,0,0,0,0,0,1,0
2,0.249589,0.873636,0.653944,0.027723,0.0,0.349511,0.294146,0.328620,0.310906,0.174676,...,0,0,0,0,0,1,0,0,0,0
3,0.112513,0.891143,0.561778,0.033706,0.0,0.358599,0.324235,0.283459,0.286912,0.151759,...,0,0,0,0,0,0,1,0,0,0
4,0.070592,0.821551,0.579708,0.040811,0.0,0.338221,0.372923,0.248045,0.319404,0.109238,...,0,0,0,0,0,1,0,0,0,0


In [21]:
y_test.head(77)

0     0
1     0
2     1
3     0
4     0
     ..
72    0
73    0
74    0
75    0
76    0
Name: hotspot, Length: 77, dtype: int32

In [24]:
# Save datasets

X_train.to_csv("../data/processed/X_train.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)

X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)